In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("../Data/train.csv")
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# Age의 NaN을 Age의 평균값으로 대체하기

train.loc[train.Age.isna(), 'Age'] = round(train.Age.mean(), 1)


# Embarked의 NaN값을 S로 대체

train.loc[train.Embarked.isna(), 'Embarked'] = "S"

In [4]:
# FamilySize 컬럼 만들기 : SibSp + Parch + 1(me)
train['FamilySize'] = train.SibSp + train.Parch + 1

train[['SibSp', 'Parch','FamilySize']].head()

,SibSp,Parch,FamilySize
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


In [5]:
# C(Chersbourg:FR), S(Southampton:UK), Q(Queentown:UK)
train['Nationality_AT'] = train.Embarked == 'C'
train['Nationality_UK'] = train.Embarked != 'C'

train[['Embarked','Nationality_AT', 'Nationality_UK']].head()

,Embarked,Nationality_AT,Nationality_UK
0,S,False,True
1,C,True,False
2,S,False,True
3,S,False,True
4,S,False,True


In [6]:
print(train.columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'FamilySize',
       'Nationality_AT', 'Nationality_UK'],
      dtype='object')


In [7]:
# Step 1: 이름에서 title 추출
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Step 2: 희귀 타이틀을 'Rare'로 통합
train['Title'] = train['Title'].replace([
    'Dr', 'Rev', 'Col', 'Major', 'Jonkheer', 'Don', 'Sir',
    'Capt', 'Countess', 'Lady', 'Dona'
], 'Rare')

# Step 3: 프랑스식 또는 대체 표현을 기존 카테고리에 합치기
train['Title'] = train['Title'].replace({
    'Mme': 'Mrs',
    'Ms': 'Miss',
    'Mlle': 'Miss'
})
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
train['Age*Class'] = train['Age'] * train['Pclass']

train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,Nationality_AT,Nationality_UK,Title,IsAlone,Age*Class
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,2,False,True,Mr,0,66.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,2,True,False,Mrs,0,38.0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1,False,True,Miss,1,78.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,2,False,True,Mrs,0,35.0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1,False,True,Mr,1,105.0


In [8]:
train.corr(numeric_only=True)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,Nationality_AT,Nationality_UK,IsAlone,Age*Class
PassengerId,1.000000,-0.005007,-0.035144,0.033206,-0.057527,-0.001652,0.012658,-0.040143,-0.001205,0.001205,0.057462,0.002815
Survived,-0.005007,1.000000,-0.338481,-0.069811,-0.035322,0.081629,0.257307,0.016639,0.168240,-0.168240,-0.203367,-0.326357
Pclass,-0.035144,-0.338481,1.000000,-0.331334,0.083081,0.018443,-0.549500,0.065997,-0.243292,0.243292,0.135207,0.531230
Age,0.033206,-0.069811,-0.331334,1.000000,-0.232624,-0.179194,0.091563,-0.248513,0.032025,-0.032025,0.179779,0.562007
SibSp,-0.057527,-0.035322,0.083081,-0.232624,1.000000,0.414838,0.159651,0.890712,-0.059528,0.059528,-0.584471,-0.183129
Parch,-0.001652,0.081629,0.018443,-0.179194,0.414838,1.000000,0.216225,0.783111,-0.011069,0.011069,-0.583398,-0.138987
Fare,0.012658,0.257307,-0.549500,0.091563,0.159651,0.216225,1.000000,0.217138,0.269335,-0.269335,-0.271832,-0.356241
FamilySize,-0.040143,0.016639,0.065997,-0.248513,0.890712,0.783111,0.217138,1.000000,-0.046215,0.046215,-0.690922,-0.194598
Nationality_AT,-0.001205,0.168240,-0.243292,0.032025,-0.059528,-0.011069,0.269335,-0.046215,1.000000,-1.000000,-0.095298,-0.191109
Nationality_UK,0.001205,-0.168240,0.243292,-0.032025,0.059528,0.011069,-0.269335,0.046215,-1.000000,1.000000,0.095298,0.191109


In [9]:
# str 컬럼 삭제
train.drop(columns=['Name', 'Ticket', 'Cabin', 'PassengerId'], inplace=True)
train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,Nationality_AT,Nationality_UK,Title,IsAlone,Age*Class
0,0,3,male,22.0,1,0,7.2500,S,2,False,True,Mr,0,66.0
1,1,1,female,38.0,1,0,71.2833,C,2,True,False,Mrs,0,38.0
2,1,3,female,26.0,0,0,7.9250,S,1,False,True,Miss,1,78.0
3,1,1,female,35.0,1,0,53.1000,S,2,False,True,Mrs,0,35.0
4,0,3,male,35.0,0,0,8.0500,S,1,False,True,Mr,1,105.0


In [10]:
train.Title.unique()

array(['Mr', 'Mrs', 'Miss', 'Master', 'Rare'], dtype=object)

In [11]:
#  범주형 데이터 숫자로 변환

# 예: 'male'/'female' → 0/1
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})


# 예: 'S', 'C', 'Q' → 0, 1, 2
train['Embarked'] = train['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})




# 숫자로 매핑
title_map = {
    'Mr': 0,
    'Miss': 1,
    'Mrs': 2,
    'Master': 3,
    'Rare': 4
}

train['Title'] = train['Title'].map(title_map).fillna(4).astype(int)


train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,Nationality_AT,Nationality_UK,Title,IsAlone,Age*Class
0,0,3,0,22.0,1,0,7.2500,0,2,False,True,0,0,66.0
1,1,1,1,38.0,1,0,71.2833,1,2,True,False,2,0,38.0
2,1,3,1,26.0,0,0,7.9250,0,1,False,True,1,1,78.0
3,1,1,1,35.0,1,0,53.1000,0,2,False,True,2,0,35.0
4,0,3,0,35.0,0,0,8.0500,0,1,False,True,0,1,105.0


In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [13]:
# Feature와 Target 분리하기

train_label = train.loc[:,'Survived']
train_input = train.iloc[:,1:]

In [14]:
train_input.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,Nationality_AT,Nationality_UK,Title,IsAlone,Age*Class
0,3,0,22.0,1,0,7.2500,0,2,False,True,0,0,66.0
1,1,1,38.0,1,0,71.2833,1,2,True,False,2,0,38.0
2,3,1,26.0,0,0,7.9250,0,1,False,True,1,1,78.0
3,1,1,35.0,1,0,53.1000,0,2,False,True,2,0,35.0
4,3,0,35.0,0,0,8.0500,0,1,False,True,0,1,105.0


In [15]:
print(train_input.shape)
print(train_label.shape)

(891, 13)
(891,)


In [16]:
# Train과 valid 나누기

train_data, valid_data, train_target, valid_target = \
  train_test_split(
    train_input,
    train_label,
    random_state=42,
    stratify=train_label,
    test_size=0.2
  )

In [17]:
print(train_data.shape)
print(train_target.shape)
print(valid_data.shape)
print(valid_target.shape)

(712, 13)
(712,)
(179, 13)
(179,)


### 랜덤포레스트

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=2000,
    max_depth=3,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(train_data, train_target)
print("Train Score:", rf.score(train_data, train_target))
print("Valid Score:", rf.score(valid_data, valid_target))

Train Score: 0.8356741573033708
Valid Score: 0.8100558659217877


In [19]:
from sklearn.model_selection import cross_validate
# train할 것이 5000개가 넘으면 Cross Validate를 안 해도 된다.

In [43]:
scores = cross_validate(
  rf,
  train_input, # X_train아니다. Cross validate는 원래 데이터를 가지고 한다.
  train_label,
  cv=20,
  return_train_score=True,
  n_jobs=-1
)

scores

{'fit_time': array([1.62446022, 1.66372705, 1.77357316, 1.68543601, 1.69431305,
        1.97316194, 2.35412192, 2.50581717, 2.2394011 , 1.85681581,
        1.94362068, 2.00683498, 2.11003089, 2.088238  , 2.16223311,
        2.02470708, 1.87233615, 1.75663424, 1.84333801, 1.65229797]),
 'score_time': array([0.07721591, 0.07730389, 0.07514811, 0.07723808, 0.0993681 ,
        0.07937121, 0.07503843, 0.06465006, 0.07569098, 0.08487105,
        0.09161615, 0.08928108, 0.07779717, 0.07587504, 0.09180689,
        0.08776689, 0.08679605, 0.09062767, 0.05244732, 0.05205894]),
 'test_score': array([0.77777778, 0.88888889, 0.77777778, 0.88888889, 0.77777778,
        0.75555556, 0.82222222, 0.93333333, 0.88888889, 0.82222222,
        0.8       , 0.77272727, 0.79545455, 0.86363636, 0.81818182,
        0.79545455, 0.77272727, 0.90909091, 0.84090909, 0.86363636]),
 'train_score': array([0.83687943, 0.83333333, 0.83451537, 0.82978723, 0.8392435 ,
        0.8392435 , 0.83451537, 0.82978723, 0.83096927,

In [44]:
print(np.mean(scores['train_score']), np.mean(scores['test_score'])) # test라고 적지만 실제로는 valid다

0.834898724743986 0.8282575757575759


In [22]:
# Test 해보기

rf.score(valid_data, valid_target)

0.8100558659217877

### xgboost

In [23]:
import warnings
warnings.filterwarnings('ignore')

In [53]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
xgb.fit(train_data, train_target)
print(xgb.score(train_data, train_target))
print(xgb.score(valid_data, valid_target))

0.8721910112359551
0.8100558659217877


In [55]:
from sklearn.model_selection import cross_val_score

CVscores = cross_val_score(xgb, train_data, train_target, cv=20, scoring='accuracy')
print("XGBoost CV 평균 정확도:", scores.mean())

XGBoost CV 평균 정확도: 0.8230555555555557


### 로지스틱

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# 특성과 라벨
X_train = train_data.copy()
y_train = train_target
X_valid = valid_data.copy()
y_valid = valid_target

# (선택) 표준화 — 로지스틱 회귀는 정규화에 민감
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

In [27]:
# 모델 정의
log_reg = LogisticRegression(max_iter=1000, random_state=42)

# 학습
log_reg.fit(X_train_scaled, y_train)

# 예측 및 평가
train_pred = log_reg.predict(X_train_scaled)
valid_pred = log_reg.predict(X_valid_scaled)

print("Train acc:", accuracy_score(y_train, train_pred))
print("Valid acc:", accuracy_score(y_valid, valid_pred))

print("\nClassification report (valid):")
print(classification_report(y_valid, valid_pred))

Train acc: 0.8202247191011236
Valid acc: 0.8100558659217877

Classification report (valid):
              precision    recall  f1-score   support

           0       0.83      0.87      0.85       110
           1       0.78      0.71      0.74        69

    accuracy                           0.81       179
   macro avg       0.80      0.79      0.80       179
weighted avg       0.81      0.81      0.81       179



---
### 앙상블

In [28]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.ensemble import VotingClassifier

In [29]:
from sklearn.ensemble import VotingClassifier

# 앙상블 모델 정의
voting_clf = VotingClassifier(
    estimators=[
        ('randomforest', rf),
        ('xgboost', xgb),
        ('logreg', log_reg),
    ],
    weights=[1, 1, 1],
    voting='hard',
)

# 학습 (스케일된 입력 사용)
voting_clf.fit(X_train_scaled, train_target)

# 평가
print("Train acc:", voting_clf.score(X_train_scaled, train_target))
print("Valid acc:", voting_clf.score(X_valid_scaled, valid_target))

Train acc: 0.8412921348314607
Valid acc: 0.8100558659217877


In [30]:
test = pd.read_csv("../Data/competition_test.csv")

test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [31]:
# Age의 NaN을 Age의 평균값으로 대체하기

test.loc[test.Age.isna(), 'Age'] = round(test.Age.mean(), 1)


# Embarked의 NaN값을 S로 대체

test.loc[test.Embarked.isna(), 'Embarked'] = "S"


# Fare의 NaN값을 0으로 대체
test.loc[test.Fare.isna(), 'Fare'] = 0

In [32]:
# FamilySize 컬럼 만들기 : SibSp + Parch + 1(me)
test['FamilySize'] = test.SibSp + test.Parch + 1

test[['SibSp', 'Parch','FamilySize']].head()

,SibSp,Parch,FamilySize
0,0,0,1
1,1,0,2
2,0,0,1
3,0,0,1
4,1,1,3


In [33]:
# C(Chersbourg:FR), S(Southampton:UK), Q(Queentown:UK)
test['Nationality_AT'] = test.Embarked == 'C'
test['Nationality_UK'] = test.Embarked != 'C'

test[['Embarked','Nationality_AT', 'Nationality_UK']].head()

,Embarked,Nationality_AT,Nationality_UK
0,Q,False,True
1,S,False,True
2,Q,False,True
3,S,False,True
4,S,False,True


In [34]:
print(test.columns)

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked', 'FamilySize', 'Nationality_AT',
       'Nationality_UK'],
      dtype='object')


In [35]:
# Step 1: 이름에서 title 추출
test['Title'] = test['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

# Step 2: 희귀 타이틀을 'Rare'로 통합
test['Title'] = test['Title'].replace([
    'Dr', 'Rev', 'Col', 'Major', 'Jonkheer', 'Don', 'Sir',
    'Capt', 'Countess', 'Lady', 'Dona'
], 'Rare')

# Step 3: 프랑스식 또는 대체 표현을 기존 카테고리에 합치기
test['Title'] = test['Title'].replace({
    'Mme': 'Mrs',
    'Ms': 'Miss',
    'Mlle': 'Miss'
})
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)
test['Age*Class'] = test['Age'] * test['Pclass']

test.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FamilySize,Nationality_AT,Nationality_UK,Title,IsAlone,Age*Class
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,1,False,True,Mr,1,103.5
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S,2,False,True,Mrs,0,141.0
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q,1,False,True,Mr,1,124.0
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S,1,False,True,Mr,1,81.0
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S,3,False,True,Mrs,0,66.0


In [36]:
# str 컬럼 삭제
test.drop(columns=['Name', 'Ticket', 'Cabin', 'PassengerId'], inplace=True)
test.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,Nationality_AT,Nationality_UK,Title,IsAlone,Age*Class
0,3,male,34.5,0,0,7.8292,Q,1,False,True,Mr,1,103.5
1,3,female,47.0,1,0,7.0000,S,2,False,True,Mrs,0,141.0
2,2,male,62.0,0,0,9.6875,Q,1,False,True,Mr,1,124.0
3,3,male,27.0,0,0,8.6625,S,1,False,True,Mr,1,81.0
4,3,female,22.0,1,1,12.2875,S,3,False,True,Mrs,0,66.0


In [37]:
#  범주형 데이터 숫자로 변환

# 예: 'male'/'female' → 0/1
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})


# 예: 'S', 'C', 'Q' → 0, 1, 2
test['Embarked'] = test['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})




# 숫자로 매핑
title_test_map = {
    'Mr': 0,
    'Miss': 1,
    'Mrs': 2,
    'Master': 3,
    'Rare': 4
}

test['Title'] = test['Title'].map(title_test_map).fillna(4).astype(int)


test.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,Nationality_AT,Nationality_UK,Title,IsAlone,Age*Class
0,3,0,34.5,0,0,7.8292,2,1,False,True,0,1,103.5
1,3,1,47.0,1,0,7.0000,0,2,False,True,2,0,141.0
2,2,0,62.0,0,0,9.6875,2,1,False,True,0,1,124.0
3,3,0,27.0,0,0,8.6625,0,1,False,True,0,1,81.0
4,3,1,22.0,1,1,12.2875,0,3,False,True,2,0,66.0


In [38]:
# 1. 필요한 feature만 추출
test_input = test

# 2. 스케일링
test_input_scaled = scaler.transform(test_input)

# 3. 예측
test_pred = voting_clf.predict(test_input_scaled)
print(test_pred)

[0 0 0 0 1 0 1 0 1 0 0 0 1 0 1 1 0 0 1 1 0 0 1 0 1 0 1 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 1 1 0 0 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 1 1 1 0 0 1 1 0 1 0 1 0
 0 1 0 1 1 0 0 0 0 0 1 1 1 1 1 0 1 0 0 0 1 0 1 0 1 0 0 0 1 0 0 0 0 0 0 1 1
 1 1 0 0 1 0 1 1 0 1 0 0 1 0 0 0 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 0 0
 1 0 0 1 1 0 1 1 1 1 0 0 0 0 1 1 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1 0 1 0 0 0 0
 0 0 1 0 1 1 0 1 1 1 1 1 0 0 1 0 1 0 0 0 0 1 0 0 1 0 1 0 1 0 1 0 1 1 0 1 0
 0 0 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 1 1 0 0
 0 0 1 0 0 0 1 1 0 1 0 0 0 0 1 1 1 1 0 0 0 0 0 0 1 0 1 0 0 1 0 0 0 0 0 0 0
 1 1 0 1 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 1 0 0 0 0 0 0 0
 0 0 1 0 1 0 1 0 1 1 0 0 0 0 1 0 0 1 0 1 1 0 1 0 1 0 0 1 0 0 1 1 0 0 0 0 0
 0 1 1 0 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 0 0 0 0 1 1 1 1 1 0 1 0 0 0]


In [39]:
test_result = pd.read_csv("../Data/competition_test_result.csv")

test_result.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [40]:
test_target = test_result['Survived']
test_target.head()

0    0
1    1
2    0
3    0
4    1
Name: Survived, dtype: int64

In [56]:
print("Train acc:", rf.score(train_data, train_target))
print("Valid acc:", rf.score(valid_data, valid_target))
print("Test  acc:", rf.score(test_input, test_target))

print("")

print("Train acc:", xgb.score(train_data, train_target))
print("Valid acc:", xgb.score(valid_data, valid_target))
print("Test  acc:", xgb.score(test_input, test_target))

print("")

print("Train acc:", log_reg.score(X_train_scaled, train_target))
print("Valid acc:", log_reg.score(X_valid_scaled, valid_target))
print("Test  acc:", log_reg.score(test_input_scaled, test_target))

print("")

print("Train acc:", voting_clf.score(X_train_scaled, train_target))
print("Valid acc:", voting_clf.score(X_valid_scaled, valid_target))
print("Test  acc:", voting_clf.score(test_input_scaled, test_target))

Train acc: 0.8356741573033708
Valid acc: 0.8100558659217877
Test  acc: 0.9631449631449631

Train acc: 0.8721910112359551
Valid acc: 0.8100558659217877
Test  acc: 0.8968058968058968

Train acc: 0.8202247191011236
Valid acc: 0.8100558659217877
Test  acc: 0.9606879606879607

Train acc: 0.8412921348314607
Valid acc: 0.8100558659217877
Test  acc: 0.9606879606879607
